# ViCAM - Viral fine-tuning of Cambriam

In [ ]:
from Bio import SeqIO

In [5]:
#input_fasta = "../data/raw/URVDBv29-prot_clustered.fasta"
input_fasta = "../data/processed/C-RVDBv29_no_poly/train.fasta"

records = list(SeqIO.parse(input_fasta, "fasta"))

#records = [record for record in records if 'poly' in record.description]

In [6]:
len(records)

568485

In [7]:
mn=0
mx=0
for record in records:
    if len(record.seq) < mn or mn == 0:
        mn = len(record.seq)
    if len(record.seq) > mx:
        mx = len(record.seq)
print(f"Min length: {mn}")
print(f"Max length: {mx}")


Min length: 11
Max length: 2047


In [3]:
count = 0
while count < 10:
    record = records[count]
    print(record.description)
    count += 1

    

acc|GENBANK|UJJ65121.1|GENBANK|OM336640|surface glycoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65122.1|GENBANK|OM336640|ORF3a protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65123.1|GENBANK|OM336640|envelope protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65124.1|GENBANK|OM336640|membrane glycoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65125.1|GENBANK|OM336640|ORF6 protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65126.1|GENBANK|OM336640|ORF7a protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65127.1|GENBANK|OM336640|ORF7b [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65128.1|GENBANK|OM336640|ORF8 protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65129.1|GENBANK|OM336640|nucleocapsid phosphoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65130.1|GENBANK|OM336640|OR

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

def filter_fasta_by_valid_amino_acids(input_fasta_path, output_fasta_path):
    """
    Filters a FASTA file to keep only sequences with valid amino acids.

    Args:
        input_fasta_path (str): Path to the input FASTA file.
        output_fasta_path (str): Path to save the filtered FASTA file.
    """
    valid_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
    
    valid_records = []
    for record in SeqIO.parse(input_fasta_path, "fasta"):
        sequence_str = str(record.seq).upper()
        is_valid = True
        for aa in sequence_str:
            if aa not in valid_amino_acids:
                is_valid = False
                print(f"Invalid character '{aa}' found in sequence: {record.id}. Skipping.")
                break
        if is_valid:
            valid_records.append(record)

    SeqIO.write(valid_records, output_fasta_path, "fasta")
    print(f"Filtered FASTA saved to: {output_fasta_path}")
    print(f"Original records: {len(list(SeqIO.parse(input_fasta_path, 'fasta')))}, Valid records: {len(valid_records)}")

# Example usage:
input_file = "../data/processed/C-RVDBv29_no_poly/train.fasta"
output_file = "../data/processed/C-RVDBv29_no_poly_20aa/train.fasta"
filter_fasta_by_valid_amino_acids(input_file, output_file)

# inference

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import torch
from esm.models.esmc import ESMC
from esm.tokenization import get_esmc_model_tokenizers

tokenizer = get_esmc_model_tokenizers()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/stor/work/Wilke/luiz/ViCAM/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
model_path= '/stor/work/Wilke/wilkelab/pLMs_checkpoints/ESMC/esmc_300m_2024_12_v0.pth'
state_dict = torch.load(model_path, map_location=device, weights_only=True)
esmc300m = ESMC(d_model=960, n_heads=15, n_layers=30, tokenizer=get_esmc_model_tokenizers())
esmc300m.load_state_dict(state_dict)
esmc300m.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esmc300m.to(device)
esmc300m

ESMC(
  (embed): Embedding(64, 960)
  (transformer): TransformerStack(
    (blocks): ModuleList(
      (0-29): 30 x UnifiedTransformerBlock(
        (attn): MultiHeadAttention(
          (layernorm_qkv): Sequential(
            (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=960, out_features=2880, bias=False)
          )
          (out_proj): Linear(in_features=960, out_features=960, bias=False)
          (q_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (k_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (rotary): RotaryEmbedding()
        )
        (ffn): Sequential(
          (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=960, out_features=5120, bias=False)
          (2): SwiGLU()
          (3): Linear(in_features=2560, out_features=960, bias=False)
        )
      )
    )
    (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
  )
  (sequ

In [3]:
for name, param in esmc300m.named_parameters():
    print(name, param.requires_grad)

embed.weight True
transformer.blocks.0.attn.layernorm_qkv.0.weight True
transformer.blocks.0.attn.layernorm_qkv.0.bias True
transformer.blocks.0.attn.layernorm_qkv.1.weight True
transformer.blocks.0.attn.out_proj.weight True
transformer.blocks.0.attn.q_ln.weight True
transformer.blocks.0.attn.k_ln.weight True
transformer.blocks.0.ffn.0.weight True
transformer.blocks.0.ffn.0.bias True
transformer.blocks.0.ffn.1.weight True
transformer.blocks.0.ffn.3.weight True
transformer.blocks.1.attn.layernorm_qkv.0.weight True
transformer.blocks.1.attn.layernorm_qkv.0.bias True
transformer.blocks.1.attn.layernorm_qkv.1.weight True
transformer.blocks.1.attn.out_proj.weight True
transformer.blocks.1.attn.q_ln.weight True
transformer.blocks.1.attn.k_ln.weight True
transformer.blocks.1.ffn.0.weight True
transformer.blocks.1.ffn.0.bias True
transformer.blocks.1.ffn.1.weight True
transformer.blocks.1.ffn.3.weight True
transformer.blocks.2.attn.layernorm_qkv.0.weight True
transformer.blocks.2.attn.layernor

In [18]:
tokens = tokenizer(["AAAAAAA", "AAAAA"], return_tensors="pt", padding=True)
tokens.to(device) 

{'input_ids': tensor([[0, 5, 5, 5, 5, 5, 5, 5, 2],
        [0, 5, 5, 5, 5, 5, 2, 1, 1]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0]], device='cuda:0')}

In [19]:
esmc300m(tokens["input_ids"])

ESMCOutput(sequence_logits=tensor([[[-38.0333, -38.0360, -38.0323,  ..., -38.0285, -38.0664, -38.0540],
         [-40.2790, -40.2907, -40.3479,  ..., -40.3142, -40.3002, -40.3140],
         [-36.3614, -36.3531, -36.3800,  ..., -36.4196, -36.3916, -36.4265],
         ...,
         [-36.4638, -36.4345, -36.5000,  ..., -36.5169, -36.4898, -36.5214],
         [-35.7786, -35.7601, -35.8030,  ..., -35.8089, -35.8304, -35.8100],
         [-33.7242, -33.7013, -33.7387,  ..., -33.7467, -33.7438, -33.7541]],

        [[-38.0487, -38.0396, -38.0528,  ..., -38.0591, -38.0906, -38.0756],
         [-40.0131, -40.0105, -40.0686,  ..., -40.0690, -40.0437, -40.0478],
         [-36.1433, -36.1190, -36.1505,  ..., -36.2211, -36.2072, -36.1873],
         ...,
         [-32.9298, -32.9026, -32.9321,  ..., -32.9482, -32.9679, -32.9473],
         [-19.2742, -19.4478, -19.2775,  ..., -19.3356, -19.1872, -19.4463],
         [-19.2742, -19.4478, -19.2775,  ..., -19.3356, -19.1872, -19.4463]]],
       device='cu

In [ ]:
path_checkpoint = "/stor/work/Wilke/luiz/ViCAM/checkpoints/ViCAM_300M/v03_no_ploy/epoch=1-val_loss=1.73.ckpt"
state_dict = torch.load(path_checkpoint)["state_dict"]
new_state_dict = {k.replace("model.", ""): v for k, v in state_dict.items()}
vicam = ESMC(d_model=960, n_heads=15, n_layers=30, tokenizer=get_esmc_model_tokenizers())
vicam.load_state_dict(new_state_dict)
vicam.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vicam.to(device)
vicam


ESMC(
  (embed): Embedding(64, 960)
  (transformer): TransformerStack(
    (blocks): ModuleList(
      (0-29): 30 x UnifiedTransformerBlock(
        (attn): MultiHeadAttention(
          (layernorm_qkv): Sequential(
            (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=960, out_features=2880, bias=False)
          )
          (out_proj): Linear(in_features=960, out_features=960, bias=False)
          (q_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (k_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (rotary): RotaryEmbedding()
        )
        (ffn): Sequential(
          (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=960, out_features=5120, bias=False)
          (2): SwiGLU()
          (3): Linear(in_features=2560, out_features=960, bias=False)
        )
      )
    )
    (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
  )
  (sequ

# Testing model embeddings

In [20]:
import torch
import pandas as pd
import numpy as np

from scipy import stats
from sklearn import metrics
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr

import warnings
warnings.filterwarnings('ignore') 
from sklearn.exceptions import ConvergenceWarning

In [21]:
def features_scaler(features):
    '''Scale the features by min-max scaler, to ensure that the features selected by Lasso are not biased by the scale of the features'''
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_features = scaler.fit_transform(features)
    return pd.DataFrame(scaled_features)


def run_regression(features, target):
    '''this version computes y_pred for train and test sets'''
    # Initialize lists for storing results
    folds, num_nonzero_coefs = [], []
    r2s_train, maes_train, rmses_train = [], [], []
    r2s_test, maes_test, rmses_test = [], [], []
    rhos_train, rhos_test = [], []

    # Define the KFold cross-validator
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    # Loop over the KFold splits
    for kfold, (train_index, test_index) in enumerate(kf.split(features)):
        # Split the data into training and testing sets
        X_train, X_test = features.iloc[train_index], features.iloc[test_index]
        y_train, y_test = target.iloc[train_index], target.iloc[test_index]

        # Define and train the regression model
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ConvergenceWarning)
            model = LassoCV(max_iter=1000, tol=1e-2, n_jobs=-1)
            model.fit(X_train, y_train)

            # get the number of non-zero coefficients
            coeficients = model.coef_
            num_nonzero_coef = np.sum(coeficients != 0)

            # Make predictions
            y_pred_train = pd.DataFrame(model.predict(X_train))
            y_pred_test = pd.DataFrame(model.predict(X_test))

            # Evaluate the model
            r2_train = metrics.r2_score(y_train, y_pred_train)
            mae_train = metrics.mean_absolute_error(y_train, y_pred_train)
            mse_train = metrics.mean_squared_error(y_train, y_pred_train)
            rmse_train = np.sqrt(mse_train)
            rho_train, p_value_train = spearmanr(y_train, y_pred_train)

            r2_test = metrics.r2_score(y_test, y_pred_test)
            mae_test = metrics.mean_absolute_error(y_test, y_pred_test)
            mse_test = metrics.mean_squared_error(y_test, y_pred_test)
            rmse_test = np.sqrt(mse_test)
            rho_test, p_value_test = spearmanr(y_test, y_pred_test)

            # Append results
            r2s_train.append(r2_train)
            maes_train.append(mae_train)
            rmses_train.append(rmse_train)
            rhos_train.append(rho_train)

            r2s_test.append(r2_test)
            maes_test.append(mae_test)
            rmses_test.append(rmse_test)
            rhos_test.append(rho_test)


            folds.append(kfold + 1)
            num_nonzero_coefs.append(num_nonzero_coef)

        # Return the collected results
        print(f"Results:  fold {kfold}, r2_train: {r2_train:.3f}, r2_test: {r2_test:.3f}, Num coefs: {num_nonzero_coef}")
    print(f"Results:  r2_train: {np.mean(r2s_train):.2f}, r2_test: {np.mean(r2s_test):.2f}, Num coefs: {np.mean(num_nonzero_coefs):.2f}")
    return r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs


## ESM C 300M

In [31]:
metadata = pd.read_csv("../data/DMS_metadata/PA_FLU_Sun2015_metadata.csv", index_col=0)  

embeddings = pd.DataFrame(torch.load('../embeddings/esmc_300m/PA_FLU_embeddings.pt')).T
embeddings.reset_index(inplace=True)
embeddings.rename(columns={"index": "ID"}, inplace=True)


data = metadata.merge(embeddings, how='inner', left_on='ID', right_on='ID')
target = data['target']
features = data.iloc[:, metadata.shape[1]:]
features = features_scaler(features)
print(features.shape)
print(target.shape)
data

(2590, 960)
(2590,)


,ID,mutant,target,sequence,0,1,2,3,4,5,...,950,951,952,953,954,955,956,957,958,959
0,PA_FLU_C8R,C8R,0.120065,MEDFVRQRFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001842,0.001954,0.010462,0.012958,0.003278,0.003426,...,0.000652,-0.012437,-0.007702,0.002752,-0.004459,0.009886,0.003699,-0.003755,-0.006035,-0.007248
1,PA_FLU_C8Y,C8Y,0.869537,MEDFVRQYFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001988,0.002019,0.010245,0.012934,0.003225,0.003475,...,0.000601,-0.012689,-0.007409,0.002945,-0.004173,0.009672,0.003122,-0.003733,-0.006044,-0.007270
2,PA_FLU_C8C,C8C,0.426665,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001767,0.001509,0.010462,0.012814,0.003647,0.003377,...,0.000815,-0.012426,-0.006838,0.002727,-0.004453,0.010092,0.003475,-0.004122,-0.006680,-0.007045
3,PA_FLU_F9L,F9L,0.119464,MEDFVRQCLNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001603,0.001408,0.010345,0.012646,0.003834,0.003286,...,0.000857,-0.012337,-0.006629,0.002766,-0.004546,0.010311,0.003645,-0.004141,-0.006735,-0.007024
4,PA_FLU_F9S,F9S,0.078119,MEDFVRQCSNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001810,0.001537,0.010888,0.012893,0.003228,0.003209,...,0.000773,-0.012788,-0.006652,0.002718,-0.004526,0.010168,0.003424,-0.003940,-0.006996,-0.007344
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2585,PA_FLU_L715L,L715L,1.394932,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001767,0.001509,0.010462,0.012814,0.003647,0.003377,...,0.000815,-0.012426,-0.006838,0.002727,-0.004453,0.010092,0.003475,-0.004122,-0.006680,-0.007045
2586,PA_FLU_L715S,L715S,0.914543,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001683,0.001725,0.010480,0.012886,0.003553,0.003232,...,0.000785,-0.012572,-0.006956,0.002753,-0.004591,0.010266,0.003608,-0.004101,-0.006722,-0.007024
2587,PA_FLU_L715L,L715L,0.446139,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001767,0.001509,0.010462,0.012814,0.003647,0.003377,...,0.000815,-0.012426,-0.006838,0.002727,-0.004453,0.010092,0.003475,-0.004122,-0.006680,-0.007045
2588,PA_FLU_R716G,R716G,0.777810,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001630,0.001480,0.010464,0.013056,0.003677,0.003463,...,0.000961,-0.012293,-0.006662,0.002908,-0.004706,0.010195,0.003459,-0.004245,-0.006762,-0.007140


In [32]:
r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs = run_regression(features, target)

print(f"Results:  r2_train: {np.mean(r2s_train):.2f}, r2_test: {np.mean(r2s_test):.2f}, Num coefs: {np.mean(num_nonzero_coefs):.2f}")

Results:  fold 0, r2_train: 0.042, r2_test: 0.048, Num coefs: 20
Results:  fold 1, r2_train: 0.024, r2_test: 0.009, Num coefs: 13
Results:  fold 2, r2_train: 0.057, r2_test: 0.019, Num coefs: 23
Results:  fold 3, r2_train: 0.055, r2_test: 0.020, Num coefs: 17
Results:  fold 4, r2_train: 0.039, r2_test: -0.004, Num coefs: 17
Results:  r2_train: 0.04, r2_test: 0.02, Num coefs: 18.00
Results:  r2_train: 0.04, r2_test: 0.02, Num coefs: 18.00


## ViCam 300M

In [33]:
metadata = pd.read_csv("../data/DMS_metadata/PA_FLU_Sun2015_metadata.csv", index_col=0)  

embeddings = pd.DataFrame(torch.load('../embeddings/vicam_300m/PA_FLU_embeddings.pt')).T
embeddings.reset_index(inplace=True)
embeddings.rename(columns={"index": "ID"}, inplace=True)


data = metadata.merge(embeddings, how='inner', left_on='ID', right_on='ID')
target = data['target']
features = data.iloc[:, metadata.shape[1]:]
features = features_scaler(features)
print(features.shape)
print(target.shape)
data

(2590, 960)
(2590,)


,ID,mutant,target,sequence,0,1,2,3,4,5,...,950,951,952,953,954,955,956,957,958,959
0,PA_FLU_C8R,C8R,0.120065,MEDFVRQRFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.007607,0.004878,0.035769,-0.002722,-0.006371,-0.003918,...,0.011980,-0.011476,0.017673,-0.019447,0.005374,-0.005222,0.003896,0.021472,-0.018110,-0.011667
1,PA_FLU_C8Y,C8Y,0.869537,MEDFVRQYFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.007479,0.004767,0.035456,-0.002771,-0.006265,-0.003827,...,0.012077,-0.011418,0.017889,-0.019058,0.005621,-0.005447,0.003862,0.021420,-0.018015,-0.011697
2,PA_FLU_C8C,C8C,0.426665,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.007660,0.004561,0.035756,-0.002565,-0.006317,-0.003660,...,0.012026,-0.011411,0.018128,-0.019320,0.005269,-0.005388,0.003568,0.021357,-0.018209,-0.011488
3,PA_FLU_F9L,F9L,0.119464,MEDFVRQCLNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.007602,0.004559,0.035717,-0.002532,-0.006188,-0.003641,...,0.012028,-0.011363,0.018198,-0.019337,0.005188,-0.005392,0.003576,0.021250,-0.018130,-0.011502
4,PA_FLU_F9S,F9S,0.078119,MEDFVRQCSNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.007882,0.004928,0.036185,-0.002666,-0.006452,-0.003573,...,0.012403,-0.011438,0.017975,-0.019588,0.005136,-0.005414,0.003570,0.021468,-0.018366,-0.011478
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2585,PA_FLU_L715L,L715L,1.394932,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.007660,0.004561,0.035756,-0.002565,-0.006317,-0.003660,...,0.012026,-0.011411,0.018128,-0.019320,0.005269,-0.005388,0.003568,0.021357,-0.018209,-0.011488
2586,PA_FLU_L715S,L715S,0.914543,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.007711,0.004816,0.035867,-0.002565,-0.006359,-0.003604,...,0.012148,-0.011257,0.018222,-0.019451,0.005354,-0.005504,0.003509,0.021507,-0.018294,-0.011498
2587,PA_FLU_L715L,L715L,0.446139,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.007660,0.004561,0.035756,-0.002565,-0.006317,-0.003660,...,0.012026,-0.011411,0.018128,-0.019320,0.005269,-0.005388,0.003568,0.021357,-0.018209,-0.011488
2588,PA_FLU_R716G,R716G,0.777810,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.007691,0.004705,0.035886,-0.002476,-0.006350,-0.003556,...,0.012208,-0.011326,0.018363,-0.019399,0.005300,-0.005570,0.003506,0.021415,-0.018324,-0.011471


In [34]:
r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs = run_regression(features, target)

print(f"Results:  r2_train: {np.mean(r2s_train):.2f}, r2_test: {np.mean(r2s_test):.2f}, Num coefs: {np.mean(num_nonzero_coefs):.2f}")

Results:  fold 0, r2_train: 0.156, r2_test: 0.175, Num coefs: 29
Results:  fold 1, r2_train: 0.177, r2_test: 0.097, Num coefs: 50
Results:  fold 2, r2_train: 0.145, r2_test: 0.198, Num coefs: 26
Results:  fold 3, r2_train: 0.180, r2_test: 0.120, Num coefs: 41
Results:  fold 4, r2_train: 0.184, r2_test: 0.107, Num coefs: 43
Results:  r2_train: 0.17, r2_test: 0.14, Num coefs: 37.80
Results:  r2_train: 0.17, r2_test: 0.14, Num coefs: 37.80
